# Email Triage GRPO Training
**Runtime → Change runtime type → T4 GPU** before running anything.

Run cells **one by one in order.**

In [ ]:
# CELL 1: Install
# Takes ~3 min. After this finishes → Runtime → Restart session → then run from Cell 2
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets
print("Install done — NOW go to Runtime → Restart session, then run from Cell 2")

### After Cell 1 finishes: **Runtime → Restart session**. Then run from Cell 2.

In [ ]:
# CELL 2: Clone repo
import os
if not os.path.exists('/content/OpenEnv'):
    !git clone https://github.com/Rhushya/OpenEnv.git /content/OpenEnv
    print('Cloned')
else:
    print('Already cloned')
os.chdir('/content/OpenEnv')

In [ ]:
# CELL 3: Setup paths
import sys
sys.path.insert(0, '/content/OpenEnv/src')
sys.path.insert(0, '/content/OpenEnv/envs')
sys.path.insert(0, '/content/OpenEnv/envs/email_triage_env')
sys.path.insert(0, '/content/OpenEnv/envs/email_triage_env/server')
print('Paths set')

In [ ]:
# CELL 4: Load model with Unsloth (4-bit, no vLLM needed)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name       = 'Qwen/Qwen2.5-1.5B',
    max_seq_length   = 512,
    dtype            = None,
    load_in_4bit     = True,
    fast_inference   = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 8,
    target_modules             = ['q_proj', 'v_proj'],
    lora_alpha                 = 8,
    lora_dropout               = 0,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
)
print('Model loaded with LoRA')

In [ ]:
# CELL 5: Reward functions
import re, sys
sys.path.insert(0, '/content/OpenEnv/envs/email_triage_env')
sys.path.insert(0, '/content/OpenEnv/envs/email_triage_env/server')

from server.email_triage_environment import EmailTriageEnvironment
from models import EmailTriageAction

def _parse(text):
    cat = re.search(r'<category>(.*?)</category>', text, re.I)
    pri = re.search(r'<priority>(\d+)</priority>', text, re.I)
    esc = re.search(r'<escalate>(true|false)</escalate>', text, re.I)
    return (
        cat.group(1).strip().lower() if cat else 'other',
        max(1, min(5, int(pri.group(1)))) if pri else 1,
        esc.group(1).lower() == 'true' if esc else False,
        bool(cat and pri and esc)
    )

def _score(prompt, completion):
    p = completion if isinstance(completion, str) else (completion[0]['content'] if isinstance(completion, list) else str(completion))
    cat, pri, esc, fmt = _parse(p)
    m = re.search(r'seed[:\s]+(\d+)', str(prompt), re.I)
    seed = int(m.group(1)) if m else 0
    try:
        env = EmailTriageEnvironment(difficulty='easy')
        env.reset(seed=seed)
        obs = env.step(EmailTriageAction(category=cat, priority=pri, should_escalate=esc))
        info = obs.info or {}
        quality = (0.5*float(info.get('category_score', 0))
                 + 0.2*float(info.get('priority_score', 0))
                 + 0.3*float(info.get('escalation_score', 0)))
    except Exception:
        quality = 0.0
    return quality, 1.0 if fmt else -1.0

def reward_quality(prompts, completions, **kw):
    return [_score(p, c)[0] for p, c in zip(prompts, completions)]

def reward_format(prompts, completions, **kw):
    return [_score(p, c)[1] for p, c in zip(prompts, completions)]

print('Reward functions ready')

In [ ]:
# CELL 6: Dataset
from datasets import Dataset

SYSTEM = (
    'You are an email triage agent. Reply ONLY with these 3 XML tags:\n'
    '<category>CATEGORY</category>\n'
    '<priority>N</priority>\n'
    '<escalate>true|false</escalate>\n'
    'Valid categories: billing support spam urgent marketing other\n'
    'Priority 1=low 5=critical'
)

EMAILS = [
    'Subject: Invoice overdue\nMy invoice #{s} is 30 days unpaid. Please resolve.',
    'Subject: Cannot login\nLocked out of account since yesterday. seed {s}',
    'Subject: Buy cheap meds\nClick here for discounts ref={s}',
    'Subject: URGENT DB breach\nProduction database compromised RIGHT NOW seed {s}',
    'Subject: Newsletter\nThanks for subscribing id={s}',
    'Subject: Refund request\nOrder {s} arrived damaged, need refund',
]

prompts = [
    [{'role': 'system', 'content': SYSTEM},
     {'role': 'user',   'content': EMAILS[i % len(EMAILS)].format(s=i)}]
    for i in range(64)
]
dataset = Dataset.from_dict({'prompt': prompts})
print(f'Dataset: {len(dataset)} prompts')

In [ ]:
# CELL 7: TRAIN
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir                  = '/content/email-triage-grpo',
    max_steps                   = 50,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations             = 4,
    max_completion_length       = 128,
    temperature                 = 0.9,
    learning_rate               = 5e-6,
    logging_steps               = 1,
    save_steps                  = 25,
    fp16                        = True,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [reward_quality, reward_format],
    train_dataset    = dataset,
    args             = config,
)

print('Starting training...')
trainer.train()
trainer.save_model('/content/email-triage-grpo')
tokenizer.save_pretrained('/content/email-triage-grpo')
print('DONE — model saved to /content/email-triage-grpo')

In [ ]:
# CELL 8: Push to HuggingFace Hub (run after training)
from huggingface_hub import HfApi

HF_TOKEN = ''            # paste your token here: hf_...
REPO_ID  = 'Rhushya/oversight-arena-grpo'

api = HfApi()
api.upload_folder(
    folder_path    = '/content/email-triage-grpo',
    repo_id        = REPO_ID,
    repo_type      = 'model',
    token          = HF_TOKEN,
    commit_message = 'GRPO Email Triage 50 steps',
)
print(f'Uploaded to https://huggingface.co/{REPO_ID}')